
# Prior predictive check: what does the model predict before it sees data?

Before fitting, sample 200 draws from the prior and push each through the
forward model. The envelope of predicted photometry is the prior predictive
distribution — what the model can produce under our chosen priors, without
any conditioning on observations.

Two things to read off the figure:

1. **Coverage.** Does the prior envelope contain the data? If not, the
   priors are inconsistent with the observation and the posterior will
   shift to the edge of prior support — a silent pathology.
2. **Width.** Where is the envelope wide vs narrow? Wide bands say "this
   parameter combination is under-constrained by the prior alone";
   narrow bands say "the prior already pins the prediction here." Bands
   that are narrow at the data point are the ones the data will *not*
   teach us much about.

This is best practice for Bayesian workflow — see Gelman et al. 2020
("Bayesian Workflow", arXiv:2011.01808 §4.2).


In [ ]:
import warnings

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

BANDS = [
    "galex_fuv",
    "galex_nuv",
    "sdss_u",
    "sdss_g",
    "sdss_r",
    "sdss_i",
    "sdss_z",
    "2mass_j",
    "2mass_h",
    "2mass_ks",
]
Z = 0.1
N_DRAWS = 200

obs = tengri.Observation(photometry=tengri.Photometry.from_names(BANDS))
ssp = tengri.load_ssp()

model = tengri.SEDModel.build(
    ssp,
    observation=obs,
    sfh={
        "type": "tsnorm",
        "*": tengri.FIXED,
        # Tight informative priors: a star-forming galaxy in the local
        # Universe with SFR ~ 0.3-10 M_sun/yr, peak ~ 1-6 Gyr ago,
        # moderate width and trunc. These make the envelope narrow enough
        # to be informative — wider priors produce a 5-dex band that
        # contains everything trivially.
        "log_total_mass": tengri.Uniform(9.6, 11.1),
        "peak_lbt_gyr": tengri.Uniform(1.0, 6.0),
        "width_gyr": tengri.Uniform(1.0, 3.0),
        "skew": 0.3,
        "trunc": 3.0,
    },
    dust={
        "type": "two_component",
        "*": tengri.FIXED,
        "tau_diff": tengri.Uniform(0.0, 0.6),
        "tau_bc": tengri.Uniform(0.0, 0.8),
    },
    redshift=tengri.Fixed(Z),
    approx=tengri.WavePrecomp(n_z=50),
)

pp = model.prior_predictive(n=N_DRAWS, seed=42)
flux_draws = np.asarray(pp.flux)  # shape (N_DRAWS, n_bands)

# Generate one mock "real" galaxy from a typical parameter set so the
# reader has a data point inside the envelope.
truth = dict(model.spec.sample(jax.random.PRNGKey(7)))
mock = model.mock(truth, snr=20.0, key=jax.random.PRNGKey(8))

wave_eff = np.array([float(jnp.mean(w)) for w in obs.photometry.filter_waves])
order = np.argsort(wave_eff)
wave_eff = wave_eff[order]
flux_draws = flux_draws[:, order]
flux_obs = np.asarray(mock.flux_obs)[order]
noise = np.asarray(mock.noise)[order]

# Per-band quantiles for the envelope.
q05, q16, q50, q84, q95 = np.percentile(flux_draws, [5, 16, 50, 84, 95], axis=0)

fig, ax = plt.subplots(figsize=(6.8, 4.2))
ax.fill_between(wave_eff, q05, q95, color="C0", alpha=0.18, label="prior 5–95 %")
ax.fill_between(wave_eff, q16, q84, color="C0", alpha=0.32, label="prior 16–84 %")
ax.plot(wave_eff, q50, color="C0", lw=1.2, label="prior median")
ax.errorbar(
    wave_eff, flux_obs, yerr=noise, fmt="o", color="k", ms=4.5, capsize=2, label="mock observation"
)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"Observed wavelength  [$\mathrm{\AA}$]")
ax.set_ylabel(r"$F_\nu$  [erg s$^{-1}$ cm$^{-2}$ Hz$^{-1}$]")
ax.legend(frameon=False, fontsize=8.5, loc="lower right")

fig.savefig("plot_prior_predictive.png", dpi=150, bbox_inches="tight")